In [2]:
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error

# Load our pre-match feature dataset
data = pd.read_csv(
    "data/processed/epl_features.csv",
    parse_dates=["Date"]
)

feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
]

# Time-based split: train on the past, test on the future
train = data[data["Season"] != "2024-25"].copy()
test = data[data["Season"] == "2024-25"].copy()

X_train = sm.add_constant(train[feature_columns])
X_test = sm.add_constant(test[feature_columns], has_constant="add")

# Train one model for home goals and one for away goals
home_goal_model = sm.GLM(
    train["FTHG"],
    X_train,
    family=sm.families.Poisson()
).fit()

away_goal_model = sm.GLM(
    train["FTAG"],
    X_train,
    family=sm.families.Poisson()
).fit()

# Predict expected goals for unseen 2024–25 matches
test["predicted_home_goals"] = home_goal_model.predict(X_test)
test["predicted_away_goals"] = away_goal_model.predict(X_test)

# Measure average error in predicted goals
home_mae = mean_absolute_error(test["FTHG"], test["predicted_home_goals"])
away_mae = mean_absolute_error(test["FTAG"], test["predicted_away_goals"])

print(f"Training matches: {len(train)}")
print(f"Test matches:     {len(test)}")
print()
print(f"Home-goal MAE: {home_mae:.3f}")
print(f"Away-goal MAE: {away_mae:.3f}")

display(
    test[
        [
            "Date", "HomeTeam", "AwayTeam",
            "FTHG", "FTAG",
            "predicted_home_goals", "predicted_away_goals"
        ]
    ].head(10)
)

Training matches: 3306
Test matches:     375

Home-goal MAE: 1.001
Away-goal MAE: 0.893


,Date,HomeTeam,AwayTeam,FTHG,FTAG,predicted_home_goals,predicted_away_goals
3306,2024-08-16,Man United,Fulham,1,0,1.628293,1.125785
3307,2024-08-17,Arsenal,Wolves,2,0,2.898090,0.683793
3308,2024-08-17,Everton,Brighton,0,3,1.846869,0.905809
3309,2024-08-17,Newcastle,Southampton,1,0,2.611428,0.877543
3310,2024-08-17,Nott'm Forest,Bournemouth,1,1,1.519546,1.223213
3311,2024-08-17,West Ham,Aston Villa,1,2,1.538157,1.363793
3312,2024-08-18,Brentford,Crystal Palace,2,1,1.166726,1.936913
3313,2024-08-18,Chelsea,Man City,0,2,1.612742,1.590675
3314,2024-08-19,Leicester,Tottenham,1,1,1.505026,1.277429
3315,2024-08-24,Brighton,Man United,2,1,1.567970,1.110407


In [3]:
from scipy.stats import poisson

# Use the first unseen 2024–25 match as an example
match = test.iloc[0]

home_team = match["HomeTeam"]
away_team = match["AwayTeam"]
home_lambda = match["predicted_home_goals"]
away_lambda = match["predicted_away_goals"]

# Calculate probabilities for scores from 0–7 goals each
score_probabilities = []

for home_goals in range(8):
    for away_goals in range(8):
        probability = (
            poisson.pmf(home_goals, home_lambda) *
            poisson.pmf(away_goals, away_lambda)
        )

        score_probabilities.append({
            "Score": f"{home_goals}-{away_goals}",
            "Probability": probability,
            "HomeGoals": home_goals,
            "AwayGoals": away_goals,
        })

score_table = pd.DataFrame(score_probabilities)

# Most likely exact scores
top_scores = score_table.sort_values(
    "Probability", ascending=False
).head(5)

# Overall match-result probabilities
home_win_probability = score_table[
    score_table["HomeGoals"] > score_table["AwayGoals"]
]["Probability"].sum()

draw_probability = score_table[
    score_table["HomeGoals"] == score_table["AwayGoals"]
]["Probability"].sum()

away_win_probability = score_table[
    score_table["HomeGoals"] < score_table["AwayGoals"]
]["Probability"].sum()

print(f"{home_team} vs {away_team}")
print(f"Expected goals: {home_team} {home_lambda:.2f} — {away_team} {away_lambda:.2f}")
print()
print(f"Home win: {home_win_probability:.1%}")
print(f"Draw:     {draw_probability:.1%}")
print(f"Away win: {away_win_probability:.1%}")
print("\nMost likely exact scores:")

display(top_scores[["Score", "Probability"]])

Man United vs Fulham
Expected goals: Man United 1.63 — Fulham 1.13

Home win: 49.0%
Draw:     24.6%
Away win: 26.3%

Most likely exact scores:


,Score,Probability
9,1-1,0.116710
8,1-0,0.103670
17,2-1,0.095019
16,2-0,0.084402
1,0-1,0.071676


In [4]:
import numpy as np

evaluation = test.copy()

# The most likely Poisson goal count is usually the whole number below the expected goals
evaluation["predicted_home_score"] = np.floor(
    evaluation["predicted_home_goals"]
).astype(int)

evaluation["predicted_away_score"] = np.floor(
    evaluation["predicted_away_goals"]
).astype(int)

# Did we predict the exact score?
evaluation["exact_score_correct"] = (
    (evaluation["FTHG"] == evaluation["predicted_home_score"]) &
    (evaluation["FTAG"] == evaluation["predicted_away_score"])
)

# Did we at least predict home win / draw / away win correctly?
evaluation["predicted_result"] = np.select(
    [
        evaluation["predicted_home_score"] > evaluation["predicted_away_score"],
        evaluation["predicted_home_score"] == evaluation["predicted_away_score"],
    ],
    ["H", "D"],
    default="A"
)

evaluation["result_correct"] = (
    evaluation["FTR"] == evaluation["predicted_result"]
)

print(f"Exact-score accuracy: {evaluation['exact_score_correct'].mean():.1%}")
print(f"Match-result accuracy: {evaluation['result_correct'].mean():.1%}")

display(
    evaluation[
        [
            "HomeTeam", "AwayTeam",
            "FTHG", "FTAG",
            "predicted_home_score", "predicted_away_score",
            "FTR", "predicted_result"
        ]
    ].head(10)
)


Exact-score accuracy: 11.2%
Match-result accuracy: 30.9%


,HomeTeam,AwayTeam,FTHG,FTAG,predicted_home_score,predicted_away_score,FTR,predicted_result
3306,Man United,Fulham,1,0,1,1,H,D
3307,Arsenal,Wolves,2,0,2,0,H,H
3308,Everton,Brighton,0,3,1,0,A,H
3309,Newcastle,Southampton,1,0,2,0,H,H
3310,Nott'm Forest,Bournemouth,1,1,1,1,D,D
3311,West Ham,Aston Villa,1,2,1,1,A,D
3312,Brentford,Crystal Palace,2,1,1,1,H,D
3313,Chelsea,Man City,0,2,1,1,A,D
3314,Leicester,Tottenham,1,1,1,1,D,D
3315,Brighton,Man United,2,1,1,1,H,D


In [5]:
from scipy.stats import poisson

def poisson_result_probabilities(home_lambda, away_lambda, max_goals=10):
    goals = np.arange(max_goals + 1)

    home_probabilities = poisson.pmf(goals, home_lambda)
    away_probabilities = poisson.pmf(goals, away_lambda)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    home_win = np.tril(score_matrix, k=-1).sum()
    draw = np.trace(score_matrix)
    away_win = np.triu(score_matrix, k=1).sum()

    return home_win, draw, away_win

# Calculate H / D / A probabilities for every test match
result_probabilities = np.array([
    poisson_result_probabilities(home_goals, away_goals)
    for home_goals, away_goals in zip(
        evaluation["predicted_home_goals"],
        evaluation["predicted_away_goals"]
    )
])

evaluation["home_win_probability"] = result_probabilities[:, 0]
evaluation["draw_probability"] = result_probabilities[:, 1]
evaluation["away_win_probability"] = result_probabilities[:, 2]

result_labels = np.array(["H", "D", "A"])
evaluation["best_probability_result"] = result_labels[
    result_probabilities.argmax(axis=1)
]

proper_result_accuracy = (
    evaluation["FTR"] == evaluation["best_probability_result"]
).mean()

print(f"Exact-score accuracy: {evaluation['exact_score_correct'].mean():.1%}")
print(f"Proper match-result accuracy: {proper_result_accuracy:.1%}")

display(
    evaluation[
        [
            "HomeTeam", "AwayTeam", "FTR",
            "home_win_probability",
            "draw_probability",
            "away_win_probability",
            "best_probability_result"
        ]
    ].head(10)
)

Exact-score accuracy: 11.2%
Proper match-result accuracy: 48.5%


,HomeTeam,AwayTeam,FTR,home_win_probability,draw_probability,away_win_probability,best_probability_result
3306,Man United,Fulham,H,0.490555,0.246101,0.263343,H
3307,Arsenal,Wolves,H,0.822618,0.117099,0.060064,H
3308,Everton,Brighton,A,0.595244,0.224244,0.180509,H
3309,Newcastle,Southampton,H,0.744713,0.152399,0.102798,H
3310,Nott'm Forest,Bournemouth,D,0.441259,0.252623,0.306117,H
3311,West Ham,Aston Villa,A,0.415560,0.246621,0.337819,H
3312,Brentford,Crystal Palace,H,0.226782,0.220244,0.552968,A
3313,Chelsea,Man City,A,0.387654,0.234113,0.378231,H
3314,Leicester,Tottenham,D,0.425718,0.251857,0.322425,H
3315,Brighton,Man United,H,0.479126,0.251531,0.269342,H
